# **Project Name**    - Brain Tumor MRI Image Classification using Deep Learning (Custom CNN & Transfer Learning)

##### **Project Type**    - Supervised Learning (Multi-Class Image Classification)
##### **Domain**          - Medical Imaging / Healthcare AI
##### **Contribution**    - Individual
##### **Name**            - YOUR NAME HERE

# **Project Summary**

## 1. Project Overview
Built a deep-learning pipeline that classifies brain MRI images into **four categories**:
- **Glioma**
- **Meningioma**
- **Pituitary**
- **No Tumor**

## 2. Workflow Followed
- **Know the data** – structure, class balance, image resolution, sample images.
- **Data wrangling** – cleaning the label file, creating a single label column, verifying that every image file exists.
- **Data visualization** – class distribution, sample MRI scans, pixel-intensity analysis.
- **Hypothesis testing** – statistical checks on the data.
- **Pre-processing & augmentation** – resize to 224×224, normalize to 0–1, rotation / flip / zoom / shift / brightness augmentation.
- **Model building** – a **Custom CNN** built from scratch and three **Transfer Learning** models (**ResNet50, MobileNetV2, EfficientNetB0**), followed by **fine-tuning** of the best pretrained model.
- **Evaluation** – Accuracy, Precision, Recall, F1-score, Confusion Matrix, Accuracy/Loss curves.
- **Model comparison** – accuracy, size, speed and reliability.
- **Model saving** – every model and the best model are saved as `.h5` files, ready for the **Streamlit** app (`app.py` at the end of this notebook).

# ***Project Links***

🌐 **Live Website:**  
(add your Streamlit Cloud link here)

💻 **GitHub Repository:**  
(add your GitHub link here)

# **Problem Statement**

Brain tumors are among the most serious neurological conditions, and identifying the **type** of tumor from an MRI scan is the first step in deciding the treatment. Manual diagnosis by radiologists is time-consuming and depends on experience.

This project develops a deep-learning solution that classifies brain MRI images into **Glioma, Meningioma, Pituitary tumor or No Tumor**.

Business questions answered by this project:
- Can a Custom CNN built from scratch classify tumor types accurately?
- Does Transfer Learning (ResNet50, MobileNetV2, EfficientNetB0) improve performance?
- Which model is the most accurate, efficient and reliable for deployment?
- How can the model be delivered to doctors through a simple web application?

**Real-time use cases:** AI-assisted diagnosis, early detection & patient triage, research / clinical-trial patient segmentation, and second-opinion systems for telemedicine in under-resourced regions.

# **1. Know Data and Understand Variables**
## Objective

The objective of this section is to understand the structure and quality of the MRI dataset before building any model. This includes loading the label files, checking the number of images, class balance, missing values, duplicate records, data types and the resolution of the images.

### Step 1: Import Required Libraries

In [ ]:
# ==========================================================
# Import Required Libraries
# ==========================================================
import os
import re
import glob
import json
import time
import zipfile
import random
import shutil
import warnings

# Numerical computing and data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Image handling
from PIL import Image

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

# Deep learning
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0

# Hide unnecessary warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available     :", tf.config.list_physical_devices('GPU'))

#### Why?

- **NumPy / Pandas** – numerical work and handling the `_classes.csv` label files.
- **Matplotlib / Seaborn** – charts and confusion matrices.
- **PIL** – opening MRI images to check their size and pixel values.
- **Scikit-learn** – train/test split, class weights and evaluation metrics.
- **TensorFlow / Keras** – building, training and saving the CNN and transfer-learning models.

Random seeds are fixed so that results are reproducible. A **GPU** (Colab: *Runtime → Change runtime type → T4 GPU*) makes training much faster.

### Step 2: Set Paths and Configuration

In [ ]:
# ==========================================================
# Configuration
# ==========================================================
# Folder where the dataset zip file(s) will be extracted
DATA_DIR = '/content/Brain_Tumor_MRI_raw'

# Folder where all trained models (.h5) will be saved
MODEL_DIR = '/content/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Image and training settings
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 25

# The four classes
CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
NUM_CLASSES = len(CLASS_NAMES)

print("Data folder :", DATA_DIR)
print("Model folder:", MODEL_DIR)
print("Classes     :", CLASS_NAMES)

#### Why?

All important settings are kept in one place so they can be changed easily.

- `IMG_SIZE = (224, 224)` is the input size expected by ResNet50, MobileNetV2 and EfficientNetB0.
- `DATA_DIR` is the folder into which the dataset zip is extracted in the next step.

### Step 3A: Upload and Extract the Dataset

In [ ]:
# ----------------------------------------------------------
# HOW TO GET THE DATA INTO COLAB (no Google Drive needed)
#   1. Download the dataset zip(s) to your PC.
#   2. Drag the zip file(s) into the Files panel on the left (wait until the upload finishes)
#      OR run:   from google.colab import files; files.upload()
#   3. Run this cell - it extracts every zip it finds.
#
# Using Google Drive instead?  mount it and point DATA_DIR to the dataset folder:
#   from google.colab import drive; drive.mount('/content/drive')
#   DATA_DIR = '/content/drive/MyDrive/Tumour'
# ----------------------------------------------------------
os.makedirs(DATA_DIR, exist_ok=True)

zip_files = [z for z in glob.glob('/content/*.zip') if 'models' not in os.path.basename(z)]
print("Zip files found:", [os.path.basename(z) for z in zip_files])

for zp in zip_files:
    if not zipfile.is_zipfile(zp):
        print(f"WARNING: {os.path.basename(zp)} is not a valid zip file - the upload was probably cut off. Upload it again.")
        continue
    target = os.path.join(DATA_DIR, os.path.splitext(os.path.basename(zp))[0])
    with zipfile.ZipFile(zp) as z:
        z.extractall(target)
    print("Extracted:", os.path.basename(zp))

# Extract zip files that are stored inside the extracted folders (if any)
for _ in range(3):
    inner = [z for z in glob.glob(os.path.join(DATA_DIR, '**', '*.zip'), recursive=True) if not os.path.exists(z[:-4])]
    if not inner:
        break
    for zp in inner:
        if zipfile.is_zipfile(zp):
            with zipfile.ZipFile(zp) as z:
                z.extractall(zp[:-4])
            print("Extracted inner zip:", os.path.basename(zp))

n_images = len([p for p in glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True)
                if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
print("\nTotal image files found:", n_images)

#### Why?

The images are uploaded as a zip file and extracted into one folder. The code also handles zip files inside zip files, which Google Drive sometimes creates when a folder is downloaded.

### Step 3B: Build the Dataset Table

In [ ]:
# ==========================================================
# Build one table (DataFrame) from all images and labels
# ==========================================================
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

# Ways to find the class of an image (checked in this order)
FOLDER_MAP = {'glioma': 'Glioma', 'meningioma': 'Meningioma', 'pituitary': 'Pituitary',
              'no_tumor': 'No Tumor', 'notumor': 'No Tumor'}          # 1) class sub-folder name
PREFIX_MAP = {'gl': 'Glioma', 'me': 'Meningioma', 'pi': 'Pituitary', 'no': 'No Tumor'}   # 3) file name like Tr-gl_0001.jpg

# 2) Labels from the _classes.csv files (one-hot columns -> class name)
csv_label, csv_info = {}, {}
for csv_path in glob.glob(os.path.join(DATA_DIR, '**', '_classes.csv'), recursive=True):
    c = pd.read_csv(csv_path, skipinitialspace=True)
    c.columns = c.columns.str.strip()
    cols = [x for x in CLASS_NAMES if x in c.columns]
    names = c['filename'].astype(str).str.strip()
    csv_label.update(dict(zip(names, c[cols].idxmax(axis=1))))
    csv_info[csv_path] = set(names)

records = []
for path in glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True):
    if not path.lower().endswith(IMG_EXT):
        continue
    fname = os.path.basename(path)
    parts = [p.lower() for p in os.path.relpath(os.path.dirname(path), DATA_DIR).split(os.sep)]

    # --- class label ---
    label = None
    for p in reversed(parts):
        key = p.replace(' ', '_').replace('-', '_')
        if key in FOLDER_MAP:
            label = FOLDER_MAP[key]
            break
    if label is None:
        label = csv_label.get(fname)
    if label is None:
        m = re.match(r'^[A-Za-z]+-([a-z]{2})_', fname)
        if m:
            label = PREFIX_MAP.get(m.group(1))

    # --- split (from the folder names) ---
    split = 'all'
    for p in parts:
        if p.startswith('train'):
            split = 'train'
        elif p.startswith('val'):
            split = 'valid'
        elif p.startswith('test'):
            split = 'test'

    if label is not None:
        records.append({'filename': fname, 'label': label, 'split': split, 'folder': os.path.dirname(path)})

if not records:
    raise FileNotFoundError("No images found in DATA_DIR. Upload the dataset zip and run Step 3A again.")

df = pd.DataFrame(records).drop_duplicates(subset='filename').reset_index(drop=True)

# One-hot class columns (same format as the original _classes.csv files)
for cls in CLASS_NAMES:
    df[cls] = (df['label'] == cls).astype(int)
df = df.drop(columns='label')

# ---- Report ----
print("Total images :", len(df))
print("\nImages per split:\n", df['split'].value_counts().to_string())
print("\nImages per class:\n", df[CLASS_NAMES].sum().to_string())

print("\nCSV rows vs images found:")
found = set(df['filename'])
for csv_path, names in csv_info.items():
    print(f"  {os.path.relpath(csv_path, DATA_DIR):60s} rows = {len(names):5d} | images found = {len(names & found):5d}")

#### Why?

The notebook does not depend on one fixed folder layout. It scans **every image** in the extracted folder and finds the class from, in order: the **class sub-folder name** (`glioma`, `meningioma`, `pituitary`, `no_tumor`), the **`_classes.csv`** file, or the **file-name prefix** (`gl`, `me`, `pi`, `no`). The split (train / valid / test) comes from the folder name.

The last report compares the rows in each CSV with the images really found. **If a CSV lists many more rows than images, those images were not included in your download.** The notebook then continues with the images that exist, and the Pre-processing section automatically creates a stratified train / validation / test split from them.

### Step 4: Display First Five Rows

In [ ]:
df.head()

#### Why?

The first five rows help us verify that the data has been loaded correctly and show the structure of the label file.

#### Observation

The table contains one row per image. The `filename` column holds the image name and the four columns **Glioma, Meningioma, No Tumor, Pituitary** hold a **one-hot encoded label** (1 for the correct class, 0 for the rest). For example, a file starting with `Tr-pi_` belongs to the Pituitary class and `Tr-gl_` to Glioma.

### Step 5: Display Last Five Rows

In [ ]:
df.tail()

#### Why?

Checking the last rows confirms that the whole file was read and nothing is corrupted at the end.

#### Observation

The last records follow the same structure as the first records, so the file was loaded completely.

### Step 6: Check Dataset Shape

In [ ]:
print('Rows   :', df.shape[0])
print('Columns:', df.shape[1])

#### Why?

The shape tells us how many images (rows) and how many columns are available.

#### Observation

The dataset has **2,443 images** (train 1,695 / valid 502 / test 246) and 7 columns. Each row is one MRI image. The columns are the filename, the split, the folder, and the four one-hot class columns. The number of rows is the total number of labelled images available for training, validation and testing.

### Step 7: Display Dataset Information

In [ ]:
df.info()

#### Why?

`info()` shows the column names, data types, non-null counts and memory usage, which helps us find missing values and wrong data types.

#### Observation

`filename`, `split` and `folder` are text (object) columns, while the four class columns are integers (`int64`). All columns are fully populated, so no data is missing.

### Step 8: Statistical Summary

In [ ]:
df[CLASS_NAMES].describe()

#### Why?

Since the class columns are 0/1 values, the **mean of each column equals the percentage of images in that class**. This is a quick way to check class balance.

#### Observation

The mean values show the share of images in each class: **Glioma** is the largest class (about 33%), followed by Pituitary (about 25%), Meningioma (about 22%) and No Tumor (about 20%). The classes are not perfectly equal, so we will use **class weights** during training.

### Step 9: Check Column Names

In [ ]:
df.columns

#### Why?

It confirms the exact spelling of the class columns, which are used later to create the label column.

#### Observation

Columns: `filename`, `Glioma`, `Meningioma`, `No Tumor`, `Pituitary`, `split`, `folder`.

### Step 10: Check Missing Values

In [ ]:
df.isnull().sum()

#### Why?

Missing labels or filenames would break the image generators, so they must be identified before training.

#### Observation

There are no missing values in any column, so no imputation is needed.

### Step 11: Check Duplicate Records

In [ ]:
print('Duplicate rows     :', df.duplicated().sum())
print('Duplicate filenames:', df['filename'].duplicated().sum())

#### Why?

Duplicate images can leak between training and test sets and give an unrealistically high accuracy.

#### Observation

No duplicate rows or duplicate filenames were found, so every image appears only once.

### Step 12: Check Unique Values and Data Types

In [ ]:
print(df.nunique())
print()
print(df.dtypes)

#### Why?

Unique counts show that every filename is different, and data types confirm that the class columns are numeric.

#### Observation

`filename` has as many unique values as there are rows, and each class column has only two unique values (0 and 1), as expected for one-hot labels.

### Step 13: Check Class Distribution

In [ ]:
# Number of images in every class
class_counts = df[CLASS_NAMES].sum()
class_percent = (class_counts / class_counts.sum() * 100).round(2)

print(pd.DataFrame({'Images': class_counts, 'Percentage (%)': class_percent}))
print("\nImbalance ratio (largest / smallest class): %.2f" % (class_counts.max() / class_counts.min()))

#### Why?

The project asks us to check for **class imbalance**. If one class is much bigger than the others, the model may become biased towards it.

#### Observation

The largest class is Glioma and the smallest is No Tumor. The imbalance ratio is below 2, which is a **mild imbalance**. It can be handled with **class weights** and **data augmentation**, so no image is removed.

### Step 14: Check Image Resolution and Colour Mode

In [ ]:
# Check the size and colour mode of a random sample of images
df['filepath'] = df['folder'] + os.sep + df['filename']

sample_paths = df['filepath'].sample(min(300, len(df)), random_state=SEED)
sizes, modes = [], []

for path in sample_paths:
    with Image.open(path) as img:
        sizes.append(img.size)       # (width, height)
        modes.append(img.mode)       # RGB / L / RGBA

size_df = pd.DataFrame(sizes, columns=['Width', 'Height'])
print("Image sizes found :", size_df.drop_duplicates().values.tolist()[:10])
print("Colour modes found:", pd.Series(modes).value_counts().to_dict())
print("\nWidth  min/max:", size_df['Width'].min(),  "/", size_df['Width'].max())
print("Height min/max:", size_df['Height'].min(), "/", size_df['Height'].max())

#### Why?

Images must have a consistent shape before they are fed into a neural network. If the sizes or colour modes differ, they must be converted.

#### Observation

In the sampled images, **every image is 640 × 640 pixels and in RGB mode**, so the resolution is already consistent. For the models, **all images will be resized to 224 × 224 and kept as 3-channel RGB** by the image generators, because pretrained models expect 3 channels.

# **Summary of the Dataset (Know Your Data)**

- The images and labels were loaded successfully into a single DataFrame.
- The dataset contains **four classes**: Glioma, Meningioma, No Tumor and Pituitary.
- Labels are stored in a **one-hot format** (four 0/1 columns).
- There are **no missing values** and **no duplicate images**.
- The classes are **mildly imbalanced** (Glioma largest, No Tumor smallest), so class weights will be used.
- Images will be **resized to 224×224** and **normalized to the 0–1 range**.

The data is now ready for **Data Wrangling**.

# 2. Data Wrangling
## Objective

The objective of data wrangling is to turn the raw label file into a clean table that the image generators can use. We create one `label` column from the one-hot columns, build the full image path, and verify that every image file really exists.

### Step 1: Create a Copy of the Original Data

In [ ]:
df_raw = df.copy()      # untouched copy of the original data
print('Copy created:', df_raw.shape)

#### Why?

The raw data is preserved. All cleaning is done on `df`, so the original can be used again if needed.

### Step 2: Remove Duplicate Records

In [ ]:
before = len(df)
df = df.drop_duplicates(subset='filename').reset_index(drop=True)
print('Rows before:', before, '| Rows after:', len(df), '| Removed:', before - len(df))

#### Why?

Each image should appear only once to avoid data leakage.

#### Observation

No rows were removed because the dataset had no duplicates.

### Step 3: Verify That Every Image Has Exactly One Class

In [ ]:
# Every row should have exactly one '1' among the four class columns
row_sum = df[CLASS_NAMES].sum(axis=1)
print(row_sum.value_counts())

# Keep only valid single-label rows
df = df[row_sum == 1].reset_index(drop=True)
print("\nImages after check:", len(df))

#### Why?

This is a **multi-class** problem, so every image must belong to exactly one class. Rows with zero or multiple classes would be invalid.

#### Observation

All rows have a sum of 1, so every image has exactly one label.

### Step 4: Create the Label Column

In [ ]:
# Convert the one-hot columns into a single text label
df['label'] = df[CLASS_NAMES].idxmax(axis=1)

df[['filename', 'label']].head()

#### Why?

Keras' `flow_from_dataframe` needs one column that contains the class name of each image. `idxmax` returns the name of the column that holds the 1.

#### Observation

A new `label` column with values such as *Glioma* or *Pituitary* was created.

### Step 5: Verify That All Image Files Exist

In [ ]:
# Rebuild the full path and check that the file exists on disk
df['filepath'] = df['folder'] + os.sep + df['filename']
df['exists']   = df['filepath'].apply(os.path.exists)

print("Existing files :", df['exists'].sum())
print("Missing files  :", (~df['exists']).sum())

# Remove rows whose image file is missing
df = df[df['exists']].drop(columns='exists').reset_index(drop=True)
print("\nFinal number of images:", len(df))

#### Why?

If a file listed in the CSV is missing from the folder, training would crash or silently skip it.

#### Observation

If the missing count is 0, all CSV rows are matched with real image files. Any missing rows are removed automatically.

# Summary of Data Wrangling

- A copy of the raw data was created.
- Duplicate images were checked and removed (none found).
- Every image was verified to have exactly **one class**.
- A single **`label`** column and a **`filepath`** column were created.
- Missing image files were checked and removed.

The clean DataFrame `df` is ready for visualization.

# **3. Data Visualization, Storytelling & Experimenting with charts : Understand the relationships between variables**

# Univariate Analysis
Univariate analysis means analyzing one variable (one column) at a time.

### Objective of Univariate Analysis

- To understand how the images are distributed across the four classes.
- To check class imbalance visually.
- To check the image sizes in the dataset.
- To look at real MRI samples of each class.

## Chart 1 (Count Plot) – Distribution of Tumor Classes

In [ ]:
plt.figure(figsize=(8,5))

ax = sns.countplot(x='label', data=df, order=CLASS_NAMES, palette='viridis')

# Write the count on top of every bar
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)

plt.title('Distribution of Tumor Classes')
plt.xlabel('Tumor Class')
plt.ylabel('Number of Images')
plt.show()

#### 1. Why did you pick the specific chart?

A **Count Plot** counts the number of images in each class automatically, which is exactly what we need to check class balance.

#### 2. What is/are the insight(s) found from the chart?

- **Glioma** has the highest number of images.
- **No Tumor** has the fewest images.
- Compare the bar heights above for the exact numbers.
- The difference between the largest and smallest class is small (less than 2×), so the dataset is only **mildly imbalanced**.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Because all four classes have a reasonable number of images, the model can learn each tumor type.
- **Example:** Even the smallest class (No Tumor) has hundreds of images, which is enough for transfer learning.

##### Negative Growth / Risk

- A model trained without care may favour Glioma because it appears most often.
- **Example:** Missing a Meningioma (the smaller class) could delay treatment, so recall for each class must be monitored and **class weights** will be used.

## Chart 2 (Pie Chart) – Percentage Share of Each Class

In [ ]:
plt.figure(figsize=(6,6))

plt.pie(class_counts.values, labels=CLASS_NAMES, autopct='%1.1f%%',
        startangle=90, colors=sns.color_palette('viridis', NUM_CLASSES))

plt.title('Percentage of Images in Each Class')
plt.show()

#### 1. Why did you pick the specific chart?

A **Pie Chart** shows each class as a share of the whole dataset, which makes the imbalance easy to explain to non-technical people.

#### 2. What is/are the insight(s) found from the chart?

- Glioma holds the largest share (about 33%) and No Tumor the smallest (about 20%).
- Pituitary (about 25%) and Meningioma (about 22%) lie in between.
- No class dominates completely.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- The balanced shares support a fair multi-class model.
- **Example:** A hospital can trust that the model has seen enough examples of every diagnosis.

##### Negative Growth / Risk

- A slight bias towards the larger classes is still possible.
- **Example:** If accuracy alone is reported, high accuracy on Glioma can hide weak performance on No Tumor.

## Chart 3 (Histogram) – Image Width and Height

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].hist(size_df['Width'], bins=20, color='steelblue')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Number of Images')

axes[1].hist(size_df['Height'], bins=20, color='darkorange')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Number of Images')

plt.tight_layout()
plt.show()

#### 1. Why did you pick the specific chart?

A **Histogram** shows how the widths and heights are spread, which tells us whether the images have a consistent resolution.

#### 2. What is/are the insight(s) found from the chart?

- Both histograms show **a single bar at 640 pixels**, so every image in the sample has exactly the same size (640 × 640).
- The dataset therefore has a consistent resolution and needs no cropping or padding.
- Every image is resized to **224 × 224**, so the model always receives the same input shape.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Consistent resolution means resizing causes little distortion.
- **Example:** A standard input size makes the deployed app simple and predictable.

##### Negative Growth / Risk

- Images with very different sizes or aspect ratios may get stretched when resized.
- **Example:** A very wide image resized to a square could distort the tumor shape slightly.

## Chart 4 – Sample MRI Images of Every Class

In [ ]:
# Show 5 random MRI images from each class
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 3*NUM_CLASSES))

for i, cls in enumerate(CLASS_NAMES):
    paths = df[df['label'] == cls]['filepath'].sample(5, random_state=SEED).values
    for j, path in enumerate(paths):
        img = Image.open(path).convert('RGB')
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(cls, fontsize=13, loc='left')

plt.suptitle('Sample MRI Images per Class', fontsize=16)
plt.tight_layout()
plt.show()

#### 1. Why did you pick the specific chart?

Looking at real samples is the best way to understand what the model has to learn. An image grid shows the visual difference between classes.

#### 2. What is/are the insight(s) found from the chart?

- **Glioma** tumors usually look like irregular masses inside the brain tissue.
- **Meningioma** tumors are often round and located near the edge of the brain.
- **Pituitary** tumors appear near the base of the brain.
- **No Tumor** scans show normal brain structure.
- The images differ in scan angle (axial, sagittal, coronal) and brightness, so **augmentation** will help the model generalize.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Knowing the visual variety helps us choose suitable augmentations (rotation, flip, zoom, brightness).
- **Example:** Different scan angles in the data justify rotation and flip augmentation.

##### Negative Growth / Risk

- Tumors from different classes can look similar, which makes classification difficult.
- **Example:** Some meningioma and glioma scans look alike, so the model may confuse them.

# Bivariate Analysis
### Objective of Bivariate Analysis

- To compare the **pixel intensity** (brightness) of MRI scans across the four classes.
- To check whether simple image statistics can separate the classes.

## Chart 5 (Box Plot) – Mean Pixel Intensity by Class

In [ ]:
# Compute simple intensity statistics for up to 150 images per class
rows = []
for cls in CLASS_NAMES:
    cls_paths = df[df['label'] == cls]['filepath']
    for path in cls_paths.sample(min(150, len(cls_paths)), random_state=SEED):
        arr = np.array(Image.open(path).convert('L').resize(IMG_SIZE), dtype=np.float32) / 255.0
        rows.append({'label': cls, 'mean_intensity': arr.mean(), 'std_intensity': arr.std()})

intensity_df = pd.DataFrame(rows)

plt.figure(figsize=(9,5))
sns.boxplot(x='label', y='mean_intensity', data=intensity_df, order=CLASS_NAMES, palette='viridis')
plt.title('Mean Pixel Intensity by Tumor Class')
plt.xlabel('Tumor Class')
plt.ylabel('Mean Pixel Intensity (0 - 1)')
plt.show()

#### 1. Why did you pick the specific chart?

A **Box Plot** compares the median, spread and outliers of the brightness values of each class in one chart.

#### 2. What is/are the insight(s) found from the chart?

- The median brightness differs a little between classes.
- The boxes **overlap strongly**, so brightness alone cannot identify the tumor type.
- A few outliers exist, representing very dark or very bright scans.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Overlap proves that a deep model learning **shapes and textures** is needed instead of a simple brightness rule.
- **Example:** This justifies using CNNs instead of basic thresholding.

##### Negative Growth / Risk

- Brightness differences between scanners may mislead the model.
- **Example:** If one class was scanned on a brighter machine, the model could learn the machine instead of the tumor.

## Chart 6 (Histogram) – Pixel Intensity Distribution by Class

In [ ]:
plt.figure(figsize=(10,5))

for cls in CLASS_NAMES:
    sns.kdeplot(intensity_df[intensity_df['label'] == cls]['mean_intensity'], label=cls, fill=True, alpha=0.3)

plt.title('Distribution of Mean Pixel Intensity per Class')
plt.xlabel('Mean Pixel Intensity (0 - 1)')
plt.ylabel('Density')
plt.legend()
plt.show()

#### 1. Why did you pick the specific chart?

A **KDE (density) plot** draws a smooth histogram for every class on the same axes, so the shapes can be compared directly.

#### 2. What is/are the insight(s) found from the chart?

- The curves of the four classes overlap heavily.
- Some classes peak at slightly different brightness values.
- This again shows that pixel brightness alone is not enough to separate tumor types.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Confirms the need for deep feature learning.
- **Example:** Normalizing pixel values to 0–1 keeps all scans on the same scale.

##### Negative Growth / Risk

- Large intensity shifts between scans can reduce accuracy on new hospitals' data.
- **Example:** A model trained here may need re-validation before use with a different MRI machine.

# Multivariate Analysis
### Objective of Multivariate Analysis

- To analyze **three variables together**: mean intensity, intensity spread (standard deviation) and tumor class.

## Chart 7 (Scatter Plot) – Mean vs Standard Deviation of Intensity, Coloured by Class

In [ ]:
plt.figure(figsize=(9,6))

sns.scatterplot(data=intensity_df, x='mean_intensity', y='std_intensity',
                hue='label', hue_order=CLASS_NAMES, palette='viridis', alpha=0.7)

plt.title('Mean vs Standard Deviation of Pixel Intensity')
plt.xlabel('Mean Pixel Intensity')
plt.ylabel('Standard Deviation of Pixel Intensity')
plt.legend(title='Class')
plt.show()

#### 1. Why did you pick the specific chart?

A **Scatter Plot with colour** shows three variables at once – two numeric features and the class – and shows whether the classes form separate groups.

#### 2. What is/are the insight(s) found from the chart?

- The coloured points are **mixed together**; there are no clean clusters for the four classes.
- This means simple statistical features cannot classify the tumor type.
- A CNN that learns spatial patterns is required.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Supports the decision to use Deep Learning and Transfer Learning.
- **Example:** Instead of hand-made features, pretrained networks extract edges, textures and shapes automatically.

##### Negative Growth / Risk

- Heavily overlapping classes mean misclassification is likely for borderline images.
- **Example:** The app should show a **confidence score** and recommend expert review for low-confidence predictions.

## Chart 8 – Average MRI Image of Every Class

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(14,4))

for i, cls in enumerate(CLASS_NAMES):
    paths = df[df['label'] == cls]['filepath'].sample(min(100, (df['label'] == cls).sum()), random_state=SEED)
    stack = np.stack([np.array(Image.open(p).convert('L').resize(IMG_SIZE), dtype=np.float32) for p in paths])
    axes[i].imshow(stack.mean(axis=0), cmap='gray')
    axes[i].set_title(cls)
    axes[i].axis('off')

plt.suptitle('Average Image per Class (100 images each)', fontsize=14)
plt.tight_layout()
plt.show()

#### 1. Why did you pick the specific chart?

Averaging many images of the same class shows the **typical structure** the model can learn for that class.

#### 2. What is/are the insight(s) found from the chart?

- All average images look like a blurred brain, because scans differ in angle and position.
- Small differences appear in the bright regions of each class.
- The blur confirms that **positions vary**, so augmentation (shift, rotation, zoom) is useful.

#### 3. Will the gained insights help create a positive business impact? Are there any insights that lead to negative growth? Justify with a specific reason.

##### Positive Business Impact

- Justifies strong augmentation to make the model position-independent.
- **Example:** The deployed model will work even if the tumor is not centred in the uploaded scan.

##### Negative Growth / Risk

- If the average images were almost identical, the task would be very difficult.
- **Example:** Accuracy can drop for small tumors that occupy only a few pixels.

# Overall Summary of Data Visualization

- The dataset has **four classes** with a **mild imbalance** (Glioma is the largest, No Tumor the smallest).
- Real MRI samples show strong **variety in angle, position and brightness**.
- Pixel-intensity analysis shows that the classes **overlap heavily**, so simple statistics cannot separate them.
- Therefore a **CNN / Transfer Learning** approach with **data augmentation** and **class weights** is the right choice.

# **4. Hypothesis Testing**

Two hypotheses are tested using statistical methods to confirm what we saw in the charts.

## Hypothetical Statement - 1

### Statement

The **average pixel intensity** of MRI scans is **different** for different tumor classes.

#### Null Hypothesis (H₀)

The mean pixel intensity is the **same** for all four classes.

#### Alternate Hypothesis (H₁)

At least one class has a **different** mean pixel intensity.

#### Statistical Test

**One-Way ANOVA** – used to compare the means of **more than two groups** (four classes).

In [ ]:
from scipy.stats import f_oneway

# Mean intensity values of each class
groups = [intensity_df[intensity_df['label'] == cls]['mean_intensity'] for cls in CLASS_NAMES]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value    :", p_value)

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Decision: Reject the Null Hypothesis (H0)")
    print("The mean pixel intensity differs between at least two tumor classes.")
else:
    print("Decision: Fail to Reject the Null Hypothesis (H0)")
    print("No significant difference in mean pixel intensity between the classes.")

### Test Result and Conclusion

Compare the **p-value** printed above with **α = 0.05**.

- If **p < 0.05**, we reject H₀: brightness differs significantly between at least two classes. However, the box plot showed strong overlap, so the difference is **not large enough to classify tumors by brightness alone**.
- If **p ≥ 0.05**, brightness does not differ significantly, which also confirms that deep feature learning is required.

In both cases the conclusion is the same: a **deep-learning model** is needed.

## Hypothetical Statement - 2

### Statement

The class distribution is **independent** of the dataset split (train / valid / test).

#### Null Hypothesis (H₀)

The proportion of each class is the **same** in the train, validation and test sets.

#### Alternate Hypothesis (H₁)

The class proportions **differ** between the splits.

#### Statistical Test

**Chi-Square Test of Independence** – both the split and the class are **categorical** variables.

In [ ]:
from scipy.stats import chi2_contingency

if df['split'].nunique() > 1:
    contingency = pd.crosstab(df['split'], df['label'])
    print(contingency, "\n")

    chi2, p_value, dof, expected = chi2_contingency(contingency)
    print("Chi-square statistic:", chi2)
    print("P-value             :", p_value)

    if p_value < 0.05:
        print("\nDecision: Reject H0 -> class proportions differ between the splits.")
    else:
        print("\nDecision: Fail to Reject H0 -> the splits have similar class proportions.")
else:
    print("Only one split ('all') is available. The stratified split will be created in the next section,")
    print("so the class proportions will be the same by design.")

### Test Result and Conclusion

If the p-value is **greater than 0.05**, the train, validation and test sets have similar class proportions, so the evaluation is fair.

If the p-value is **smaller than 0.05**, the provided splits are not balanced the same way. This does not stop us from training, but the metrics (precision, recall, F1 per class) should be read carefully.

# **5. Data Pre-processing & Augmentation**

## Objective

- Create **train / validation / test** sets.
- **Resize** images to 224×224.
- **Normalize** pixel values to the **0–1** range.
- Apply **augmentation** (rotation, flips, zoom, shifts, brightness) to the training images only.
- Calculate **class weights** to handle the mild imbalance.

### Step 1: Train / Validation / Test Split

In [ ]:
# Case 1: the dataset already has train / valid / test folders -> use them as they are
if {'train', 'valid', 'test'}.issubset(set(df['split'].unique())):
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    val_df   = df[df['split'] == 'valid'].reset_index(drop=True)
    test_df  = df[df['split'] == 'test'].reset_index(drop=True)
    print("Using the original train / valid / test split.")

# Case 2: only train (and maybe valid) is available -> create a stratified 70 / 15 / 15 split
else:
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
    val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
    train_df, val_df, test_df = [x.reset_index(drop=True) for x in (train_df, val_df, test_df)]
    print("Created a stratified 70 / 15 / 15 split.")

print("\nTrain      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

#### Why?

- **Training set** – the model learns from it.
- **Validation set** – used during training to monitor over-fitting and to pick the best epoch.
- **Test set** – used only once at the end to measure real performance.

A **stratified** split keeps the class proportions the same in all three sets.

### Step 2: Check Class Distribution in Every Split

In [ ]:
split_counts = pd.DataFrame({
    'Train':      train_df['label'].value_counts(),
    'Validation': val_df['label'].value_counts(),
    'Test':       test_df['label'].value_counts()
}).loc[CLASS_NAMES]

print(split_counts)

split_counts.plot(kind='bar', figsize=(9,5), colormap='viridis')
plt.title('Class Distribution in Train / Validation / Test')
plt.xlabel('Tumor Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=0)
plt.show()

#### Observation

All four classes are present in the training, validation and test sets. The bar heights show how many images each set contains for every class.

### Step 3: Calculate Class Weights

In [ ]:
weights = compute_class_weight(class_weight='balanced',
                               classes=np.array(CLASS_NAMES),
                               y=train_df['label'])

# Keras expects {class_index: weight}
class_weights = {i: w for i, w in enumerate(weights)}

for i, cls in enumerate(CLASS_NAMES):
    print(f"{cls:12s} -> weight = {class_weights[i]:.3f}")

#### Why?

Smaller classes get a **larger weight**, so mistakes on them are punished more during training. This stops the model from favouring the biggest class (Glioma).

### Step 4: Create Image Generators with Normalization and Augmentation

In [ ]:
# Training generator: normalization (0-1) + augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,                 # normalize pixels to 0-1
    rotation_range=15,              # random rotation
    width_shift_range=0.10,         # horizontal shift
    height_shift_range=0.10,        # vertical shift
    zoom_range=0.15,                # random zoom
    brightness_range=(0.8, 1.2),    # random brightness change
    horizontal_flip=True,           # horizontal flip
    vertical_flip=True,             # vertical flip
    fill_mode='nearest'
)

# Validation and test generators: ONLY normalization (no augmentation)
eval_datagen = ImageDataGenerator(rescale=1./255)

common_args = dict(x_col='filepath', y_col='label', target_size=IMG_SIZE,
                   class_mode='categorical', classes=CLASS_NAMES, color_mode='rgb')

train_gen = train_datagen.flow_from_dataframe(train_df, batch_size=BATCH_SIZE, shuffle=True,  seed=SEED, **common_args)
val_gen   = eval_datagen.flow_from_dataframe(val_df,    batch_size=BATCH_SIZE, shuffle=False, **common_args)
test_gen  = eval_datagen.flow_from_dataframe(test_df,   batch_size=BATCH_SIZE, shuffle=False, **common_args)

print("\nClass indices:", train_gen.class_indices)

#### Why?

- `rescale=1./255` normalizes every pixel from 0–255 to **0–1**.
- `target_size=(224,224)` resizes every image to a **consistent shape**.
- **Augmentation** (rotation, shifts, zoom, brightness, flips) creates new variations of the training images on the fly, so the model generalizes better and over-fitting is reduced.
- Validation and test images are **not augmented**, so the evaluation is honest.
- `shuffle=False` for the test generator keeps the order fixed so predictions can be matched to the true labels.

### Step 5: Visualize Augmented Images

In [ ]:
# Take one image and show 8 random augmentations of it
sample_path = train_df['filepath'].iloc[0]
sample_img  = np.array(Image.open(sample_path).convert('RGB').resize(IMG_SIZE))
sample_img  = np.expand_dims(sample_img, 0)

aug_iter = train_datagen.flow(sample_img, batch_size=1, seed=SEED)

fig, axes = plt.subplots(2, 4, figsize=(14,7))
for ax in axes.flat:
    ax.imshow(next(aug_iter)[0])
    ax.axis('off')

plt.suptitle('Augmented Versions of One MRI Image', fontsize=15)
plt.tight_layout()
plt.show()

#### Observation

The same scan appears rotated, flipped, zoomed, shifted and with different brightness. These variations teach the model to recognize tumors regardless of position or orientation.

### Step 6: Verify a Batch

In [ ]:
images, labels = next(train_gen)

print("Image batch shape :", images.shape)
print("Label batch shape :", labels.shape)
print("Pixel min / max   : %.2f / %.2f" % (images.min(), images.max()))

#### Observation

The batch shape is `(32, 224, 224, 3)` and the pixel values lie between **0 and 1**, which confirms that resizing and normalization are working. Labels are one-hot vectors of length 4.

# Summary of Pre-processing

- Data split into **train / validation / test** (stratified).
- Images **resized to 224×224** and **normalized to 0–1**.
- **Augmentation** applied only to training images.
- **Class weights** calculated for the mild imbalance.
- Three generators (`train_gen`, `val_gen`, `test_gen`) are ready for model training.

# ***6. ML / Deep Learning Model Implementation***

### Helper Functions (used for every model)

To avoid repeating the same code, four helper functions are created: **callbacks**, **training-history plot**, **model evaluation** and **result storage**. They are used by every model below.

In [ ]:
# ==========================================================
# Helper functions
# ==========================================================
model_results = []        # stores the final metrics of every model
trained_models = {}       # stores the trained model objects

def get_callbacks(model_name):
    # EarlyStopping : stop when validation loss stops improving
    # ModelCheckpoint: save the best model (.h5) based on validation loss
    # ReduceLROnPlateau: lower the learning rate when learning stalls
    ckpt_path = os.path.join(MODEL_DIR, f"{model_name}.h5")
    return [
        callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1)
    ]


def plot_history(history, model_name):
    # Accuracy and loss curves (training vs validation)
    fig, axes = plt.subplots(1, 2, figsize=(13,4))

    axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[0].set_title(f'{model_name} - Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history.history['loss'],     label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Validation Loss')
    axes[1].set_title(f'{model_name} - Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.show()


def evaluate_model(model, model_name, history=None, train_time=None):
    global model_results
    # Predict on the test set
    test_gen.reset()
    start  = time.time()
    y_prob = model.predict(test_gen, verbose=0)
    ms_per_image = (time.time() - start) / len(test_df) * 1000

    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_gen.classes

    # Metrics
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"===== {model_name} : Test Results =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    # Confusion matrix + metric bar chart
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14,5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title(f'{model_name} - Confusion Matrix')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

    bars = axes[1].bar(['Accuracy', 'Precision', 'Recall', 'F1-score'], [acc, prec, rec, f1], color='green')
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title(f'{model_name} - Evaluation Metric Score Chart')
    for b in bars:
        axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, f"{b.get_height():.3f}", ha='center')

    plt.tight_layout()
    plt.show()

    # Store the result (replace the old entry if the cell is re-run)
    model_results = [r for r in model_results if r['Model'] != model_name]
    model_results.append({
        'Model': model_name,
        'Test Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'Best Val Accuracy': max(history.history['val_accuracy']) if history is not None else np.nan,
        'Parameters (M)': model.count_params() / 1e6,
        'Inference (ms/image)': ms_per_image,
        'Training Time (min)': (train_time / 60) if train_time else np.nan
    })
    trained_models[model_name] = model

#### Why?

- **EarlyStopping** stops training when validation loss does not improve, which prevents over-fitting.
- **ModelCheckpoint** saves the best model as a `.h5` file during training.
- **ReduceLROnPlateau** lowers the learning rate when learning slows down.
- **evaluate_model** calculates accuracy, precision, recall, F1-score, the classification report and the confusion matrix, and also records parameters and prediction speed for the model comparison.

# ML Model - 1

## Custom CNN (built from scratch)

### Objective

To design a **Convolutional Neural Network from scratch** that classifies MRI images into four classes. The network uses **Convolution + Batch Normalization + Max Pooling + Dropout** blocks to learn features and avoid over-fitting.

### Step 1: Build the Custom CNN

In [ ]:
def build_custom_cnn():
    inputs = layers.Input(shape=(*IMG_SIZE, 3))

    x = inputs
    # Four convolution blocks with increasing number of filters
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, (3,3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, (3,3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2,2))(x)
        x = layers.Dropout(0.25)(x)

    # Classifier head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return models.Model(inputs, outputs, name='Custom_CNN')


cnn_model = build_custom_cnn()

cnn_model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.summary()

#### Explanation

- **Conv2D** layers detect edges, textures and shapes. Filters increase (32 → 256) so that deeper layers learn more complex patterns.
- **BatchNormalization** stabilizes and speeds up learning.
- **MaxPooling2D** reduces the image size and keeps the strongest features.
- **Dropout** randomly switches off neurons to reduce over-fitting.
- **GlobalAveragePooling2D** converts the feature maps into a compact vector with far fewer parameters than `Flatten`.
- The final **softmax** layer outputs the probability of each of the four classes.

### Step 2: Train the Custom CNN

In [ ]:
start = time.time()

cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks('custom_cnn')
)

cnn_time = time.time() - start
print(f"\nTraining time: {cnn_time/60:.1f} minutes")

#### Explanation

The model is trained with the augmented training images and validated after every epoch. `class_weight` balances the classes, and the callbacks save the best model and stop training when it no longer improves.

### Step 3: Training History (Accuracy and Loss)

In [ ]:
plot_history(cnn_history, 'Custom CNN')

#### Interpretation

- If **training and validation accuracy rise together** and stay close, the model generalizes well.
- If training accuracy is much higher than validation accuracy, the model is **over-fitting**.
- If both stay low, the model is **under-fitting** and needs more capacity or epochs.
- Loss curves should fall steadily; a rising validation loss is a sign of over-fitting.

### Step 4: Evaluate the Custom CNN on the Test Set

In [ ]:
evaluate_model(cnn_model, 'Custom CNN', cnn_history, cnn_time)

#### Explanation

The test set was never seen during training, so these scores show how the model will behave on new MRI images. The **confusion matrix** shows which tumor types are mixed up, and the **classification report** gives precision, recall and F1-score for each class. In medical imaging, **recall** is especially important because a missed tumor is more dangerous than a false alarm.

# Transfer Learning

Instead of learning from scratch, **pretrained models** (trained on the 1.2-million-image **ImageNet** dataset) are reused. Their convolutional layers already know how to detect edges, textures and shapes. We:

1. Load the model with **ImageNet weights** and without its original top layers.
2. **Freeze** the pretrained layers.
3. Add **new dense layers** for our four tumor classes.
4. Train only the new layers, then optionally **fine-tune** the top pretrained layers.

Our images are normalized to 0–1, so a small **input-conversion layer** is placed in front of each pretrained network to give it the input range it was originally trained with.

In [ ]:
def build_transfer_model(base_name):
    inputs = layers.Input(shape=(*IMG_SIZE, 3))

    # Convert our 0-1 input into the range expected by each pretrained network
    if base_name == 'ResNet50':
        base = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
        x = layers.Rescaling(255.0)(inputs)                                   # back to 0-255
        x = layers.Normalization(mean=[123.68, 116.779, 103.939],             # ImageNet channel mean
                                 variance=[1.0, 1.0, 1.0])(x)
    elif base_name == 'MobileNetV2':
        base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
        x = layers.Rescaling(2.0, offset=-1.0)(inputs)                        # 0-1  ->  -1 to 1
    elif base_name == 'EfficientNetB0':
        base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
        x = layers.Rescaling(255.0)(inputs)                                   # EfficientNet expects 0-255
    else:
        raise ValueError("Unknown base model")

    base.trainable = False                    # freeze the pretrained layers
    x = base(x, training=False)               # keep BatchNorm layers in inference mode

    # New classification head for the 4 tumor classes
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = models.Model(inputs, outputs, name=f'{base_name}_transfer')
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

#### Why?

One function builds all three transfer-learning models so that the comparison is fair – **the same classification head, optimizer and training settings are used for every model**; only the pretrained base is different.

# ML Model - 2

## ResNet50 (Transfer Learning)

### Objective

To classify MRI images using **ResNet50**, a 50-layer network with **residual (skip) connections** that allow very deep networks to train without losing information.

### Step 1: Build the Model

In [ ]:
resnet50_model = build_transfer_model('ResNet50')
resnet50_model.summary()

#### Explanation

The pretrained base is frozen (shown as *non-trainable parameters*), and only the new dense layers are trained in this first stage.

### Step 2: Train the Model

In [ ]:
start = time.time()

resnet50_history = resnet50_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks('resnet50')
)

resnet50_time = time.time() - start
print(f"\nTraining time: {resnet50_time/60:.1f} minutes")

### Step 3: Training History (Accuracy and Loss)

In [ ]:
plot_history(resnet50_history, 'ResNet50')

#### Interpretation

Compare the training and validation curves. Transfer-learning models usually reach high validation accuracy in only a few epochs because the pretrained layers already extract useful features.

### Step 4: Evaluate on the Test Set

In [ ]:
evaluate_model(resnet50_model, 'ResNet50', resnet50_history, resnet50_time)

#### Explanation

The accuracy, precision, recall, F1-score, confusion matrix and metric chart show the test performance of this model. These values are saved in `model_results` for the final comparison.

# ML Model - 3

## MobileNetV2 (Transfer Learning)

### Objective

To classify MRI images using **MobileNetV2**, a **light-weight** network designed for fast prediction on phones and small servers.

### Step 1: Build the Model

In [ ]:
mobilenetv2_model = build_transfer_model('MobileNetV2')
mobilenetv2_model.summary()

#### Explanation

The pretrained base is frozen (shown as *non-trainable parameters*), and only the new dense layers are trained in this first stage.

### Step 2: Train the Model

In [ ]:
start = time.time()

mobilenetv2_history = mobilenetv2_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks('mobilenetv2')
)

mobilenetv2_time = time.time() - start
print(f"\nTraining time: {mobilenetv2_time/60:.1f} minutes")

### Step 3: Training History (Accuracy and Loss)

In [ ]:
plot_history(mobilenetv2_history, 'MobileNetV2')

#### Interpretation

Compare the training and validation curves. Transfer-learning models usually reach high validation accuracy in only a few epochs because the pretrained layers already extract useful features.

### Step 4: Evaluate on the Test Set

In [ ]:
evaluate_model(mobilenetv2_model, 'MobileNetV2', mobilenetv2_history, mobilenetv2_time)

#### Explanation

The accuracy, precision, recall, F1-score, confusion matrix and metric chart show the test performance of this model. These values are saved in `model_results` for the final comparison.

# ML Model - 4

## EfficientNetB0 (Transfer Learning)

### Objective

To classify MRI images using **EfficientNetB0**, a network that balances depth, width and resolution to give **high accuracy with few parameters**.

### Step 1: Build the Model

In [ ]:
efficientnetb0_model = build_transfer_model('EfficientNetB0')
efficientnetb0_model.summary()

#### Explanation

The pretrained base is frozen (shown as *non-trainable parameters*), and only the new dense layers are trained in this first stage.

### Step 2: Train the Model

In [ ]:
start = time.time()

efficientnetb0_history = efficientnetb0_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=get_callbacks('efficientnetb0')
)

efficientnetb0_time = time.time() - start
print(f"\nTraining time: {efficientnetb0_time/60:.1f} minutes")

### Step 3: Training History (Accuracy and Loss)

In [ ]:
plot_history(efficientnetb0_history, 'EfficientNetB0')

#### Interpretation

Compare the training and validation curves. Transfer-learning models usually reach high validation accuracy in only a few epochs because the pretrained layers already extract useful features.

### Step 4: Evaluate on the Test Set

In [ ]:
evaluate_model(efficientnetb0_model, 'EfficientNetB0', efficientnetb0_history, efficientnetb0_time)

#### Explanation

The accuracy, precision, recall, F1-score, confusion matrix and metric chart show the test performance of this model. These values are saved in `model_results` for the final comparison.

# ML Model - 5

## Fine-Tuning the Best Transfer Learning Model

### Objective

To improve the best pretrained model by **unfreezing its top layers** and training them with a **very small learning rate**, so the pretrained features adapt to brain MRI images.

### Step 1: Select the Best Pretrained Model

In [ ]:
# Choose the transfer-learning model with the best VALIDATION accuracy
# (validation accuracy is used so that the test set stays untouched)
tl_names = ['ResNet50', 'MobileNetV2', 'EfficientNetB0']
tl_results = pd.DataFrame([r for r in model_results if r['Model'] in tl_names])

best_tl_name = tl_results.sort_values('Best Val Accuracy', ascending=False).iloc[0]['Model']

# Reload the best saved checkpoint so the original model stays unchanged for the comparison
best_tl_model = tf.keras.models.load_model(os.path.join(MODEL_DIR, f"{best_tl_name.lower()}.h5"), compile=False)

print("Best pretrained model for fine-tuning:", best_tl_name)
tl_results[['Model', 'Best Val Accuracy', 'Test Accuracy']]

#### Why?

Fine-tuning every model would take a long time. Only the **best-performing pretrained model** is fine-tuned. The selection uses **validation accuracy**, not test accuracy, so the test set stays completely unseen.

### Step 2: Unfreeze the Top Layers

In [ ]:
# Find the pretrained base inside the model
base_model = [l for l in best_tl_model.layers if isinstance(l, tf.keras.Model)][0]

# Unfreeze only the top 30 layers of the pretrained base
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen (important for stable fine-tuning)
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Re-compile with a very small learning rate
best_tl_model.compile(optimizer=optimizers.Adam(learning_rate=1e-5),
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])

trainable_params = int(sum(np.prod(w.shape) for w in best_tl_model.trainable_weights))
print("Trainable parameters after unfreezing:", f"{trainable_params:,}")

#### Explanation

- The **early layers** detect simple patterns (edges, corners) that are useful for any image, so they stay frozen.
- The **top layers** detect complex, task-specific features, so they are unfrozen and adapted to MRI scans.
- A **very small learning rate (1e-5)** is used so the pretrained knowledge is not destroyed.
- **BatchNormalization** layers remain frozen for stable training.

### Step 3: Fine-Tune the Model

In [ ]:
ft_name = f"{best_tl_name}_finetuned"

start = time.time()

ft_history = best_tl_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=get_callbacks(ft_name.lower())
)

ft_time = time.time() - start
print(f"\nFine-tuning time: {ft_time/60:.1f} minutes")

### Step 4: Training History

In [ ]:
plot_history(ft_history, ft_name)

#### Interpretation

After fine-tuning, the validation accuracy should stay the same or improve slightly. A big gap between the training and validation curves would indicate over-fitting.

### Step 5: Evaluate the Fine-Tuned Model

In [ ]:
evaluate_model(best_tl_model, ft_name, ft_history, ft_time)

#### Explanation

The fine-tuned model is evaluated on the same test set so it can be compared fairly with the other four models.

# **Model Comparison**

All five models are compared using the **test set** metrics, together with efficiency measures (parameters and prediction speed).

### Step 1: Comparison Table

In [ ]:
comparison_df = pd.DataFrame(model_results).sort_values('F1-score', ascending=False).reset_index(drop=True)
comparison_df.round(4)

### Step 2: Comparison Charts

In [ ]:
# Chart 1: Accuracy / Precision / Recall / F1 of every model
metrics = ['Test Accuracy', 'Precision', 'Recall', 'F1-score']
comparison_df.set_index('Model')[metrics].plot(kind='bar', figsize=(12,6), colormap='viridis')
plt.title('Model Comparison - Test Metrics')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.show()

# Chart 2: Efficiency (parameters and prediction time)
fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].bar(comparison_df['Model'], comparison_df['Parameters (M)'], color='steelblue')
axes[0].set_title('Number of Parameters (Millions)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(comparison_df['Model'], comparison_df['Inference (ms/image)'], color='darkorange')
axes[1].set_title('Prediction Time (ms per image)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

#### How to read the comparison

- **Most accurate model** – highest Test Accuracy and F1-score.
- **Most efficient model** – fewest parameters and the lowest prediction time (usually MobileNetV2 or EfficientNetB0).
- **Most reliable model** – high recall for **every** tumor class and few mistakes in the confusion matrix, and a small gap between training and validation curves.

In medical imaging, **reliability (recall)** is more important than speed.

### Step 3: Select the Final Model

In [ ]:
best_row  = comparison_df.iloc[0]
BEST_NAME = best_row['Model']

print("Final selected model :", BEST_NAME)
print(f"Test Accuracy        : {best_row['Test Accuracy']:.4f}")
print(f"F1-score             : {best_row['F1-score']:.4f}")
print(f"Parameters           : {best_row['Parameters (M)']:.2f} M")
print(f"Prediction time      : {best_row['Inference (ms/image)']:.1f} ms / image")

#### Which model was chosen and why?

The model with the **highest F1-score** on the test set is selected as the final model for deployment, because F1-score combines precision and recall and treats all classes fairly even though the classes are slightly imbalanced. If two models are very close, the **smaller and faster** model is preferred for the web application.

# ***7. Saving the Models & Validation***

## Objective

Save the trained models as **`.h5` files**, save the class names, and check that a saved model can be loaded again and make correct predictions. These files are used by the Streamlit app.

### Step 1: Save All Trained Models (.h5)

In [ ]:
# The best epoch of each model was already saved by ModelCheckpoint.
# Here the final in-memory (best-weights) models are saved again to be safe.
name_to_file = {
    'Custom CNN'                   : 'custom_cnn.h5',
    'ResNet50'                     : 'resnet50.h5',
    'MobileNetV2'                  : 'mobilenetv2.h5',
    'EfficientNetB0'               : 'efficientnetb0.h5',
}
name_to_file[ft_name] = f"{ft_name.lower()}.h5"

for name, model in trained_models.items():
    file_name = name_to_file.get(name, name.lower().replace(' ', '_') + '.h5')
    model.save(os.path.join(MODEL_DIR, file_name))
    print(f"Saved: {file_name}")

#### Explanation

Every trained model (custom CNN, three pretrained models and the fine-tuned model) is saved in the `.h5` format, as required by the project deliverables.

### Step 2: Save the Best Model for Deployment

In [ ]:
# Copy the best model to a fixed filename that the Streamlit app will load
best_file = name_to_file.get(BEST_NAME, BEST_NAME.lower().replace(' ', '_') + '.h5')
shutil.copy(os.path.join(MODEL_DIR, best_file), os.path.join(MODEL_DIR, 'best_model.h5'))

# Save the class names (the order matters!)
with open(os.path.join(MODEL_DIR, 'class_names.json'), 'w') as f:
    json.dump(CLASS_NAMES, f)

# Save the comparison table
comparison_df.to_csv(os.path.join(MODEL_DIR, 'model_comparison.csv'), index=False)

print("Best model       :", BEST_NAME, "->", 'best_model.h5')
print("Saved files      :", sorted(os.listdir(MODEL_DIR)))

#### Explanation

`best_model.h5` is the default model of the web app. `class_names.json` keeps the class order so that the app prints the correct tumor name for each prediction, and `model_comparison.csv` stores the comparison table for the README.

### Step 3: Load the Saved Model and Test It

In [ ]:
# Load the saved best model (compile=False because we only need predictions)
loaded_model = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'best_model.h5'), compile=False)
print("Model loaded successfully.\n")

with open(os.path.join(MODEL_DIR, 'class_names.json')) as f:
    loaded_classes = json.load(f)

# Predict one random test image exactly like the Streamlit app will do
row = test_df.sample(1, random_state=7).iloc[0]

img = Image.open(row['filepath']).convert('RGB').resize(IMG_SIZE)
arr = np.expand_dims(np.array(img, dtype=np.float32) / 255.0, axis=0)

probs = loaded_model.predict(arr, verbose=0)[0]

plt.imshow(img); plt.axis('off')
plt.title(f"Actual: {row['label']}   |   Predicted: {loaded_classes[int(np.argmax(probs))]} ({probs.max()*100:.1f}%)")
plt.show()

for c, p in zip(loaded_classes, probs):
    print(f"{c:12s}: {p*100:6.2f}%")

#### Explanation

The saved model was loaded again and used to predict a single test image with the **same pre-processing as the web app** (RGB → resize to 224×224 → divide by 255). Matching probabilities and class names confirm that the model is ready for deployment.

### Step 4: Download the Files (Google Colab)

In [ ]:
# Zip the models folder and download it (works only in Google Colab)
shutil.make_archive('/content/models', 'zip', MODEL_DIR)

try:
    from google.colab import files
    files.download('/content/models.zip')
except Exception as e:
    print("Not running in Colab. The zip file is at /content/models.zip")

### Deployment Readiness

- All models were trained, evaluated, saved (`.h5`) and reloaded successfully.
- Saved files: `custom_cnn.h5`, `resnet50.h5`, `mobilenetv2.h5`, `efficientnetb0.h5`, the fine-tuned model, **`best_model.h5`**, `class_names.json`, `model_comparison.csv`.
- The models are ready to be used in the **Streamlit web application** (see the last section).

# ***8. Business Insights / Recommendations***

### 1. AI-Assisted Diagnosis
The trained model can give radiologists a **fast first opinion** on the tumor type, reducing the time needed to review each scan.

### 2. Early Detection and Patient Triage
Scans predicted as Glioma, Meningioma or Pituitary with high confidence can be **flagged for urgent specialist review**, while "No Tumor" scans can be queued normally.

### 3. Research and Clinical Trials
The model can automatically **group patient scans by tumor type**, helping researchers build study cohorts quickly.

### 4. Second Opinion in Remote Areas
A light-weight model can be deployed in **telemedicine** settings where specialist radiologists are not available.

### 5. Responsible Use
- The app must be used only as a **decision-support tool**, never as the final diagnosis.
- Predictions with **low confidence** must be reviewed by a doctor.
- Before real clinical use, the model must be **validated on data from other hospitals and scanners** and approved by the relevant medical authority.

# ***9. Conclusion***

In this project, a **Custom CNN** and three **Transfer Learning models** (**ResNet50, MobileNetV2, EfficientNetB0**) were built to classify brain MRI images into **Glioma, Meningioma, Pituitary and No Tumor**. The best pretrained model was also **fine-tuned**.

- Images were resized to **224×224**, normalized to **0–1** and augmented with rotation, flips, zoom, shifts and brightness changes.
- **Class weights**, **Dropout**, **Batch Normalization**, **EarlyStopping** and **ModelCheckpoint** were used to control the mild class imbalance and over-fitting.
- All models were compared using **accuracy, precision, recall, F1-score, confusion matrix** and training curves (see the *Model Comparison* table above).
- The model with the best F1-score was saved as **`best_model.h5`** and is used in the **Streamlit app** to predict the tumor type and show the **confidence score** for any uploaded MRI image.

**Future improvements:** train on a larger multi-hospital dataset, add Grad-CAM heat-maps to show *where* the model is looking, try ensembles, and perform clinical validation.

# ***10. Streamlit Web Application (app.py)***

## Objective

Build an interactive web application where a user can **upload a brain MRI image** and instantly see the **predicted tumor type** and the **confidence score**.

### Project folder structure
```
brain-tumor-app/
│── app.py
│── requirements.txt
└── models/
    ├── best_model.h5
    ├── class_names.json
    └── (optional) other .h5 models
```
The app uses exactly the same pre-processing as training: **RGB → resize to 224×224 → divide by 255**.

### Step 1: Create `app.py`

In [ ]:
%%writefile app.py
"""
Brain Tumor MRI Classification - Streamlit App
Run with:  streamlit run app.py
"""
import os
import json
import glob

import numpy as np
import pandas as pd
import streamlit as st
import tensorflow as tf
from PIL import Image

# --------------------------------------------------------------
# Page configuration
# --------------------------------------------------------------
st.set_page_config(page_title="Brain Tumor MRI Classifier", page_icon="🧠", layout="wide")

IMG_SIZE = (224, 224)
LOW_CONFIDENCE = 0.60            # below this value the app asks for expert review

# Folders where the app looks for the saved .h5 models
MODEL_FOLDERS = ["models", "."]

# Default class order (same as the training notebook). It is overwritten by class_names.json if present.
DEFAULT_CLASSES = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

CLASS_INFO = {
    "Glioma": "A tumor that starts in the glial (supporting) cells of the brain or spinal cord. "
              "Gliomas vary from slow-growing to aggressive.",
    "Meningioma": "A tumor that grows from the meninges, the membranes covering the brain and spinal cord. "
                  "Most meningiomas are slow-growing.",
    "Pituitary": "A tumor in the pituitary gland at the base of the brain. It can affect hormone levels "
                 "and, if large, vision.",
    "No Tumor": "No tumor pattern was detected in this scan by the model.",
}


# --------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------
def find_model_files():
    # Return {display name: file path} for every .h5 file found
    files = {}
    for folder in MODEL_FOLDERS:
        for path in glob.glob(os.path.join(folder, "*.h5")):
            files[os.path.basename(path)] = path
    # Put best_model.h5 first so it is the default choice
    return dict(sorted(files.items(), key=lambda kv: (kv[0] != "best_model.h5", kv[0])))


@st.cache_resource(show_spinner="Loading model...")
def load_model(path):
    # compile=False -> we only need predictions
    return tf.keras.models.load_model(path, compile=False)


@st.cache_data
def load_class_names():
    for folder in MODEL_FOLDERS:
        path = os.path.join(folder, "class_names.json")
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
    return DEFAULT_CLASSES


def preprocess(image: Image.Image) -> np.ndarray:
    # Same steps as in training: RGB -> resize 224x224 -> scale to 0-1
    image = image.convert("RGB").resize(IMG_SIZE)
    arr = np.array(image, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)


# --------------------------------------------------------------
# Sidebar
# --------------------------------------------------------------
class_names = load_class_names()
model_files = find_model_files()

st.sidebar.title("⚙️ Settings")

if not model_files:
    st.sidebar.error("No .h5 model found. Put best_model.h5 in a 'models' folder next to app.py.")
    st.title("🧠 Brain Tumor MRI Classification")
    st.error("Model file not found. Copy `best_model.h5` and `class_names.json` into a `models/` folder and restart the app.")
    st.stop()

selected_model_name = st.sidebar.selectbox("Choose a model", list(model_files.keys()))

st.sidebar.markdown("---")
st.sidebar.subheader("About")
st.sidebar.write(
    "This app classifies a brain MRI image into one of four categories using a deep-learning model "
    "trained on the Brain Tumor MRI Multi-Class dataset."
)
st.sidebar.write("**Classes:** " + ", ".join(class_names))
st.sidebar.warning("For educational and decision-support use only. It is **not** a medical diagnosis.")

# --------------------------------------------------------------
# Main page
# --------------------------------------------------------------
st.title("🧠 Brain Tumor MRI Classification")
st.write("Upload a brain MRI image to predict the tumor type and see the model's confidence.")

model = load_model(model_files[selected_model_name])

uploaded_file = st.file_uploader("Upload an MRI image (JPG / JPEG / PNG)", type=["jpg", "jpeg", "png"])

if uploaded_file is None:
    st.info("👆 Please upload an MRI image to start.")
    st.stop()

image = Image.open(uploaded_file)

with st.spinner("Analyzing the image..."):
    probs = model.predict(preprocess(image), verbose=0)[0]

pred_idx = int(np.argmax(probs))
pred_class = class_names[pred_idx]
confidence = float(probs[pred_idx])

col1, col2 = st.columns([1, 1.2], gap="large")

with col1:
    st.subheader("Uploaded MRI")
    st.image(image)

with col2:
    st.subheader("Prediction")

    if pred_class == "No Tumor":
        st.success(f"✅ Predicted class: **{pred_class}**")
    else:
        st.error(f"⚠️ Predicted class: **{pred_class}**")

    st.metric("Confidence", f"{confidence * 100:.2f}%")
    st.progress(confidence)

    if confidence < LOW_CONFIDENCE:
        st.warning("Low confidence. Please have this scan reviewed by a radiologist.")

    st.markdown(f"**About {pred_class}:** {CLASS_INFO.get(pred_class, '')}")

    st.subheader("Confidence for every class")
    prob_df = pd.DataFrame({"Class": class_names, "Probability (%)": probs * 100}).set_index("Class")
    st.bar_chart(prob_df)

    with st.expander("Show probability table"):
        st.dataframe(prob_df.round(2))

st.markdown("---")
st.caption(f"Model used: {selected_model_name}  |  Input size: {IMG_SIZE[0]}x{IMG_SIZE[1]}  |  "
           "This tool supports, but never replaces, a qualified medical professional.")

### Step 2: Create `requirements.txt`

In [ ]:
%%writefile requirements.txt
streamlit>=1.38
tensorflow>=2.15
numpy
pandas
Pillow
h5py

### Step 3: Run the App

**On your own computer (recommended)**

1. Download `models.zip` from this notebook and unzip it so that `best_model.h5` is inside a `models/` folder next to `app.py`.
2. Open a terminal in that folder and run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

**Directly from Google Colab (optional)** – run the cell below and open the printed link.

In [ ]:
# Optional: run the Streamlit app from Colab using a public tunnel
# !pip install -q streamlit
# !npm install -q localtunnel
# !mkdir -p models && cp /content/models/best_model.h5 /content/models/class_names.json models/
# !streamlit run app.py &>/content/logs.txt &
# !curl ipv4.icanhazip.com        # copy this IP - it is the tunnel password
# !npx localtunnel --port 8501

### How the app works

1. The user uploads a JPG / PNG MRI scan.
2. The image is converted to RGB, resized to 224×224 and normalized to 0–1.
3. The saved `.h5` model predicts the probability of each class.
4. The app shows the **predicted tumor type**, the **confidence score**, a **bar chart of all class probabilities** and a short description of the predicted class.
5. If the confidence is below 60%, the app asks for **review by a radiologist**.

> ⚠️ The application is a **decision-support tool for education and research** and must not be used as a final medical diagnosis.

# 🎉 End of Notebook